# SC03 Student Performance and Academic Intervention System
## Step 2: Data Pipeline, Quality Inspection, and Reproducible Splitting

This notebook serves as official verification evidence for Step 2:
1. **Part A:** Inspect raw student performance dataset.
2. **Part B:** Clean missing values/duplicates, enforce validation contract, and generate stratified 70/15/15 splits.
3. **Part C:** Generate and validate 10% testing-only scenarios with safeguards.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "student_performance_raw.csv"
print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw File: {RAW_FILE}")


Project Root: C:\Users\Abhishek\Capstone Project
Raw File: C:\Users\Abhishek\Capstone Project\data\raw\student_performance_raw.csv


In [2]:
df = pd.read_csv(RAW_FILE)

print("SC03 STUDENT PERFORMANCE DATA INSPECTION")
print("-" * 45)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print("\nRisk label class distribution:")
print(df['risk_label'].value_counts().sort_index())
print("\nFirst five rows:")
print(df.head())


SC03 STUDENT PERFORMANCE DATA INSPECTION
---------------------------------------------
Dataset shape: 12005 rows, 7 columns

Column names:
['record_id', 'attendance_pct', 'assessment_pct', 'assignment_pct', 'engagement_score', 'prior_performance_pct', 'risk_label']

Missing values per column:
record_id                 0
attendance_pct           12
assessment_pct           12
assignment_pct           12
engagement_score         12
prior_performance_pct    12
risk_label                0

Duplicate rows: 5

Risk label class distribution:
risk_label
high      3527
low       5339
medium    3139

First five rows:
   record_id  attendance_pct  assessment_pct  assignment_pct  engagement_score  prior_performance_pct risk_label
0  STU-00001            63.8            67.3            62.3               7.1                   55.8     medium
1  STU-00002            79.3            54.0            55.8               5.0                   37.0     medium
2  STU-00003            51.9            67.8  

## Step 2B: Clean the dataset without changing the original raw file

Cleaning operates on an in-memory copy (`clean_df = df.copy()`). Missing continuous indicators are imputed with column medians, and exact duplicate rows are removed.


In [3]:
FEATURE_COLUMNS = [
    "attendance_pct",
    "assessment_pct",
    "assignment_pct",
    "engagement_score",
    "prior_performance_pct"
]

clean_df = df.copy()

missing_before = clean_df.isna().sum()
duplicates_before = int(clean_df.duplicated().sum())

print("Missing values before cleaning:")
print(missing_before)
print(f"\nDuplicate rows before cleaning: {duplicates_before}")


Missing values before cleaning:
record_id                 0
attendance_pct           12
assessment_pct           12
assignment_pct           12
engagement_score         12
prior_performance_pct    12
risk_label                0

Duplicate rows before cleaning: 5


In [4]:
clean_df[FEATURE_COLUMNS] = clean_df[FEATURE_COLUMNS].fillna(
    clean_df[FEATURE_COLUMNS].median()
)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("DATA CLEANING SUMMARY")
print("-" * 45)
print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning: {len(clean_df)}")
print(f"Duplicate rows removed: {duplicates_before}")


DATA CLEANING SUMMARY
---------------------------------------------
Rows before cleaning: 12005
Rows after cleaning: 12000
Duplicate rows removed: 5


In [5]:
print("Missing values after cleaning:")
print(clean_df.isna().sum())


Missing values after cleaning:
record_id                0
attendance_pct           0
assessment_pct           0
assignment_pct           0
engagement_score         0
prior_performance_pct    0
risk_label               0


In [6]:
print("First five cleaned rows:")
print(clean_df.head())


First five cleaned rows:
   record_id  attendance_pct  assessment_pct  assignment_pct  engagement_score  prior_performance_pct risk_label
0  STU-00001            63.8            67.3            62.3               7.1                   55.8     medium
1  STU-00002            79.3            54.0            55.8               5.0                   37.0     medium
2  STU-00003            51.9            67.8            53.2               6.9                   77.5     medium
3  STU-00004            60.1            67.3            52.5               2.9                   82.6     medium
4  STU-00005            63.5            30.7            34.5               2.4                   57.3       high


## Step 2C: Validate the cleaned dataset

Enforce the data validation contract to ensure no invalid, out-of-range, or corrupted records proceed to model training.


In [7]:
required_columns = [
    "record_id",
    "attendance_pct",
    "assessment_pct",
    "assignment_pct",
    "engagement_score",
    "prior_performance_pct",
    "risk_label"
]

numeric_columns = [
    "attendance_pct",
    "assessment_pct",
    "assignment_pct",
    "engagement_score",
    "prior_performance_pct"
]

validation_results = {
    "Missing required columns": len(
        [col for col in required_columns if col not in clean_df.columns]
    ),
    "Missing values remaining": int(clean_df[required_columns].isna().sum().sum()),
    "Percentage values outside 0 to 100": int(
        (~clean_df[["attendance_pct", "assessment_pct", "assignment_pct", "prior_performance_pct"]].apply(lambda s: s.between(0, 100))).sum().sum()
    ),
    "Engagement scores outside 0 to 10": int((~clean_df["engagement_score"].between(0, 10)).sum()),
    "Invalid risk_label values": int((~clean_df["risk_label"].isin(["low", "medium", "high"])).sum()),
    "Duplicate rows remaining": int(clean_df.duplicated().sum())
}

print("CLEANED DATA VALIDATION")
print("-" * 45)
for check, count in validation_results.items():
    print(f"{check}: {count}")


CLEANED DATA VALIDATION
---------------------------------------------
Missing required columns: 0
Missing values remaining: 0
Percentage values outside 0 to 100: 0
Engagement scores outside 0 to 10: 0
Invalid risk_label values: 0
Duplicate rows remaining: 0


In [8]:
if sum(validation_results.values()) == 0:
    print("VALIDATION PASSED: Cleaned data is ready for splitting.")
else:
    print("VALIDATION NEEDS ATTENTION: Fix the checks with non-zero values.")


VALIDATION PASSED: Cleaned data is ready for splitting.


In [9]:
print("Data types:")
print(clean_df.dtypes)
print("\nRisk label class distribution after cleaning:")
print(clean_df['risk_label'].value_counts().sort_index())


Data types:
record_id                 object
attendance_pct           float64
assessment_pct           float64
assignment_pct           float64
engagement_score         float64
prior_performance_pct    float64
risk_label                object

Risk label class distribution after cleaning:
risk_label
high      3526
low       5339
medium    3135


In [10]:
SEED = 42
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

def create_stratified_splits(dataframe):
    train_parts = []
    validation_parts = []
    test_parts = []
    
    for label, group in dataframe.groupby("risk_label", sort=True):
        shuffled_group = group.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        total_rows = len(shuffled_group)
        test_rows = round(total_rows * TEST_RATIO)
        val_rows = round(total_rows * VALIDATION_RATIO)
        
        test_parts.append(shuffled_group.iloc[:test_rows])
        validation_parts.append(shuffled_group.iloc[test_rows : test_rows + val_rows])
        train_parts.append(shuffled_group.iloc[test_rows + val_rows :])
        
    train_data = pd.concat(train_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    validation_data = pd.concat(validation_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    test_data = pd.concat(test_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    
    return train_data, validation_data, test_data

train_df, validation_df, test_df = create_stratified_splits(clean_df)

assert len(train_df) + len(validation_df) + len(test_df) == len(clean_df)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(PROCESSED_DIR / "student_performance_clean.csv", index=False)
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
validation_df.to_csv(PROCESSED_DIR / "validation.csv", index=False)
validation_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("DATA SPLIT AND SAVE SUMMARY")
print("-" * 45)
print(f"Cleaned dataset:    {len(clean_df):,} rows")
print(f"Training dataset:   {len(train_df):,} rows")
print(f"Validation dataset: {len(validation_df):,} rows")
print(f"Test dataset:       {len(test_df):,} rows")
print(f"Random seed used:   {SEED}")


DATA SPLIT AND SAVE SUMMARY
---------------------------------------------
Cleaned dataset:    12,000 rows
Training dataset:   8,400 rows
Validation dataset: 1,800 rows
Test dataset:       1,800 rows
Random seed used:   42


In [11]:
print("Class distribution in each split:")
for split_name, split_data in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df),
]:
    print(f"\n{split_name}:")
    print(split_data['risk_label'].value_counts().sort_index())

print("\nSaved files in data/processed:")
for file_path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"- {file_path.name}")


Class distribution in each split:

Training:
risk_label
high      2468
low       3737
medium    2195

Validation:
risk_label
high      529
low       801
medium    470

Test:
risk_label
high      529
low       801
medium    470

Saved files in data/processed:
- dataset.csv
- protected_attributes.csv
- student_performance_clean.csv
- test.csv
- train.csv
- val.csv
- validation.csv


## Part C: Generated Testing Scenarios

The cited benchmark dataset contains 12,000 real records.
For testing, boundary analysis, and demonstration, we create 1,200 generated testing scenarios (10% ratio).

These scenarios are only for product testing and demonstration. They are not new real student observations and must never be mixed into training, validation, or final test data.


In [12]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from generate_test_scenarios import run_generator

scenarios = run_generator()


SC03 GENERATED TESTING SCENARIOS COMPLETED
---------------------------------------------
Real cleaned records: 12,000
Generated testing scenarios: 1,200
Generation seed: 42
Output file: C:\Users\Abhishek\Capstone Project\data\generated\generated_student_performance_test_scenarios.csv
Report file: C:\Users\Abhishek\Capstone Project\data\reports\generated_scenarios_report.md
Scenario table shape: (1200, 7)

Expected class balance:
expected_risk_label
medium    400
high      400
low       400

First five generated testing scenarios:
 scenario_id  attendance_pct  assessment_pct  assignment_pct  engagement_score  prior_performance_pct expected_risk_label
GEN-STU-0001            69.9            55.4            77.0               3.7                   50.3              medium
GEN-STU-0002            85.1            46.6            32.5               3.9                   88.9              medium
GEN-STU-0003            54.9            84.9            65.2               4.1                   9